# 04｜歷史日型基準與站點尖峰
1–17 日訓練，18–24 日驗證。排除 1/1、1/9 聯邦停班與 1/20；候選早窗 06–10、晚窗 15–20（右端不含）。各站取平均借車最多的連續兩小時，同分選較早者。保留原始時序計算 lag。

In [1]:
from zipfile import ZipFile
import pandas as pd
start, split, end = map(pd.Timestamp, ['2025-01-01', '2025-01-18', '2025-01-25'])
with ZipFile('../data/raw/202501-capitalbikeshare-tripdata.zip') as z:
    with z.open('202501-capitalbikeshare-tripdata.csv') as f:
        rides = pd.read_csv(f, usecols=['started_at', 'start_station_id'], dtype='string')
hour = pd.to_datetime(rides.started_at, format='mixed', errors='coerce').dt.floor('h')
station = rides.start_station_id.str.strip().replace('', pd.NA)
events = pd.DataFrame({'station': station, 'hour': hour})
events = events.loc[hour.ge(start) & hour.lt(end) & station.notna()]
stations = sorted(events.loc[events.hour.lt(split) & ~events.hour.dt.day.isin([1, 9]), 'station'].unique())
cohort = events[events.station.isin(stations)]
grid = pd.MultiIndex.from_product(
    [stations, pd.date_range(start, end, freq='h', inclusive='left')],
    names=['station', 'hour'])
panel = cohort.groupby(['station', 'hour']).size().reindex(grid, fill_value=0).rename('actual').to_frame()
assert panel.actual.sum() == len(cohort)


In [2]:
p = panel.reset_index()
p['lag'] = p.groupby('station').actual.shift(1)
p['date'] = p.hour.dt.normalize()
p['h'] = p.hour.dt.hour
p['weekend'] = p.hour.dt.dayofweek >= 5
ordinary = p.loc[~p.hour.dt.day.isin([1, 9, 20])]
train = ordinary.loc[ordinary.hour < split]
valid = ordinary.loc[ordinary.hour >= split].copy()
keys = ['station', 'weekend', 'h']
valid = valid.join(train.groupby(keys).actual.mean().rename('history'), on=keys)
assert valid[['history', 'lag']].notna().all().all()
weekday = train.loc[~train.weekend]
def score(label, frame):
    row = [label, frame.station.nunique(), len(frame)]
    for pred in ['lag', 'history']:
        gap = frame.actual - frame[pred]
        row.extend([gap.abs().mean(), gap.clip(lower=0).mean()])
    return row
rows = [score('all', valid)]
for label, lo, hi in [('AM', 6, 10), ('PM', 15, 20)]:
    window = weekday.loc[weekday.h.between(lo, hi-1)]
    days = window.loc[window.actual.gt(0)].groupby('station').date.nunique()
    hourly = window.groupby(['station','h']).actual.mean().unstack()
    pairs = hourly.iloc[:, :-1] + hourly.shift(-1, axis=1).iloc[:, :-1]
    peak = pairs.idxmax(axis=1)
    for threshold in (3, 5, 7):
        eligible = days[days.ge(threshold)].index
        v = valid.loc[~valid.weekend & valid.station.isin(eligible)]
        first = v.station.map(peak)
        rows.append(score(f'{label}/{threshold}', v.loc[(v.h == first) | (v.h == first + 1)]))
report = pd.DataFrame(rows, columns=['scope','stations','n','lag_MAE','lag_under','hist_MAE','hist_under'])
print(report.round(4).to_string(index=False))

scope  stations      n  lag_MAE  lag_under  hist_MAE  hist_under
  all       748 107712   0.3663     0.1831    0.3066      0.1661
 AM/3       451   3608   0.9593     0.6547    0.7731      0.4534
 AM/5       357   2856   1.1541     0.7906    0.9046      0.5401
 AM/7       269   2152   1.3866     0.9545    1.0534      0.6417
 PM/3       529   4232   1.0248     0.6099    0.8414      0.4456
 PM/5       453   3624   1.1598     0.6912    0.9414      0.5006
 PM/7       382   3056   1.3171     0.7876    1.0485      0.5663


低估量為 max(實際−預測, 0)，以所有入選格平均。零值沿用站點覆蓋假設。尖峰樣本因門檻而改變。